<a href="https://colab.research.google.com/github/br05758135-cell/gerador-documentos-disciplinares/blob/main/teste_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

numero_secreto = random.randint(1, 10)

while True:
    tentativa = int(input("Adivinhe um número de 1 a 10: "))

    if tentativa == numero_secreto:
        print("🎉 Você acertou!")
        break
    elif tentativa < numero_secreto:
        print("⬆️ Mais alto!")
    else:
        print("⬇️ Mais baixo!")

Adivinhe um número de 1 a 10: 9
⬇️ Mais baixo!
Adivinhe um número de 1 a 10: 4
⬆️ Mais alto!
Adivinhe um número de 1 a 10: 7
⬇️ Mais baixo!
Adivinhe um número de 1 a 10: 5
🎉 Você acertou!


In [3]:
import random

opcoes = ["pedra", "papel", "tesoura"]

jogador = input("Escolha pedra, papel ou tesoura: ").lower()
computador = random.choice(opcoes)

print("Computador escolheu:", computador)

if jogador == computador:
    print("Empate!")
elif (
    (jogador == "pedra" and computador == "tesoura") or
    (jogador == "papel" and computador == "pedra") or
    (jogador == "tesoura" and computador == "papel")
):
    print("Você venceu!")
else:
    print("Você perdeu!")

Escolha pedra, papel ou tesoura: Pedra
Computador escolheu: tesoura
Você venceu!


In [ ]:
from IPython.display import HTML

HTML("""
<style>
  body { background:#050509; }
  #gameContainer {
    width:960px;
    margin:10px auto;
    border:3px solid #2a224a;
    box-shadow:0 0 40px #1b0f42;
    background:#050509;
  }
  canvas { display:block; background:#07070d; }
  #info {
    width:960px;
    margin:8px auto;
    color:#ddd;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.6;
  }
  kbd {
    background:#222;
    border:1px solid #555;
    border-radius:4px;
    padding:2px 6px;
  }
</style>

<div id="gameContainer">
  <canvas id="game" width="960" height="540"></canvas>
</div>

<div id="info">
  <b>Controles:</b>
  <kbd>A</kbd>/<kbd>D</kbd> mover |
  <kbd>W</kbd>/<kbd>Espaço</kbd> pular |
  <kbd>J</kbd> espada |
  <kbd>K</kbd> dash |
  <kbd>U</kbd> Amaterasu |
  <kbd>I</kbd> Kirin |
  <kbd>O</kbd> Susanoo |
  <kbd>R</kbd> reiniciar
</div>

<script>
const canvas = document.getElementById("game");
const ctx = canvas.getContext("2d");

const WIDTH = canvas.width;
const HEIGHT = canvas.height;

let keys = {};
let pressed = {};
let camera = { x:0, y:0 };

document.addEventListener("keydown", e => {
  const k = e.key.toLowerCase();
  if (!keys[k]) pressed[k] = true;
  keys[k] = true;

  if (e.key === " ") {
    if (!keys[" "]) pressed[" "] = true;
    keys[" "] = true;
    e.preventDefault();
  }

  if (k === "r") resetGame();
});

document.addEventListener("keyup", e => {
  const k = e.key.toLowerCase();
  keys[k] = false;
  if (e.key === " ") keys[" "] = false;
});

const gravity = 0.65;
const friction = 0.82;

let world = { width:3300, height:740 };

let player;
let enemies;
let platforms;
let spikes;
let chakraOrbs;
let particles;
let slashEffects;
let blackFlames;
let lightningBolts;
let damageTexts;
let leaves;

let gameOver = false;
let victory = false;
let stormFlash = 0;

let bossCutscene = {
  active:false,
  done:false,
  timer:0,
  textIndex:0
};

function rr(x,y,w,h,r,color){
  ctx.fillStyle = color;
  if (ctx.roundRect) {
    ctx.beginPath();
    ctx.roundRect(x,y,w,h,r);
    ctx.fill();
  } else {
    ctx.fillRect(x,y,w,h);
  }
}

function resetGame(){
  gameOver = false;
  victory = false;
  stormFlash = 0;

  bossCutscene.active = false;
  bossCutscene.done = false;
  bossCutscene.timer = 0;
  bossCutscene.textIndex = 0;

  player = {
    x:80,
    y:290,
    w:38,
    h:66,
    vx:0,
    vy:0,
    speed:0.82,
    maxSpeed:5.5,
    jumpPower:-13.5,
    grounded:false,
    canDoubleJump:true,
    facing:1,

    hp:10,
    maxHp:10,
    chakra:60,
    maxChakra:100,

    susanooEnergy:0,
    susanooMax:100,
    susanooActive:false,
    susanooTimer:0,
    susanooDuration:520,

    invincible:0,
    attackCooldown:0,
    attacking:0,
    dashCooldown:0,
    dashing:0,
    amaterasuCooldown:0,
    kirinCooldown:0
  };

  platforms = [
    {x:0,y:460,w:640,h:90},
    {x:720,y:415,w:280,h:44},
    {x:1090,y:365,w:300,h:44},
    {x:1480,y:460,w:440,h:90},
    {x:2010,y:400,w:280,h:44},
    {x:2360,y:460,w:420,h:90},
    {x:2860,y:420,w:430,h:130},

    {x:350,y:340,w:170,h:28},
    {x:780,y:285,w:150,h:28},
    {x:1210,y:250,w:170,h:28},
    {x:1620,y:335,w:150,h:28},
    {x:2160,y:275,w:175,h:28},
    {x:2580,y:325,w:155,h:28}
  ];

  spikes = [
    {x:655,y:488,w:60,h:30},
    {x:1400,y:488,w:70,h:30},
    {x:1935,y:488,w:70,h:30},
    {x:2790,y:488,w:60,h:30}
  ];

  enemies = [
    createZetsu(455, 410, 370, 580, "zetsu"),
    createZetsu(810, 363, 730, 960, "fast"),
    createZetsu(1160, 315, 1095, 1370, "zetsu"),
    createZetsu(1640, 410, 1500, 1900, "spore"),
    createZetsu(2145, 350, 2020, 2280, "fast"),
    createZetsu(2460, 410, 2370, 2760, "spore"),
    createZetsu(3000, 332, 2880, 3240, "boss")
  ];

  chakraOrbs = [];
  particles = [];
  slashEffects = [];
  blackFlames = [];
  lightningBolts = [];
  damageTexts = [];

  leaves = [];
  for(let i=0;i<80;i++){
    leaves.push({
      x:Math.random()*world.width,
      y:Math.random()*HEIGHT,
      vx:-0.25-Math.random()*0.35,
      vy:0.15+Math.random()*0.25,
      size:2+Math.random()*3,
      color:Math.random()>0.5 ? "#6a8d3d" : "#9a6734"
    });
  }
}

function createZetsu(x,y,minX,maxX,type){
  const data = {
    zetsu:{w:40,h:58,hp:4,speed:1.1},
    fast:{w:38,h:56,hp:3,speed:1.8},
    spore:{w:44,h:62,hp:5,speed:1.05},
    boss:{w:92,h:116,hp:24,speed:1.08}
  }[type];

  return {
    x,y,
    w:data.w,
    h:data.h,
    hp:data.hp,
    maxHp:data.hp,
    vx:data.speed,
    minX,
    maxX,
    type,
    alive:true,
    hurt:0,
    burn:0,
    sporeCooldown:100 + Math.random()*70,
    susanooHitCooldown:0
  };
}

function rectsCollide(a,b){
  return a.x < b.x+b.w &&
         a.x+a.w > b.x &&
         a.y < b.y+b.h &&
         a.y+a.h > b.y;
}

function spawnParticles(x,y,color,count=12,power=5){
  for(let i=0;i<count;i++){
    particles.push({
      x,y,
      vx:(Math.random()-0.5)*power,
      vy:(Math.random()-0.85)*power,
      life:30+Math.random()*26,
      color,
      size:2+Math.random()*3
    });
  }
}

function addDamageText(x,y,text,color){
  damageTexts.push({x,y,text,color,life:45,vy:-1.15});
}

function gainSusanooEnergy(amount){
  player.susanooEnergy = Math.min(player.susanooMax, player.susanooEnergy + amount);
}

function damageEnemy(enemy, amount, color, knockback=true){
  if(!enemy.alive) return;

  enemy.hp -= amount;
  enemy.hurt = 12;

  if(knockback) enemy.x += player.facing * 16;

  addDamageText(enemy.x, enemy.y-8, "-" + amount, color);
  spawnParticles(enemy.x+enemy.w/2, enemy.y+enemy.h/2, color, 16, 5);

  player.chakra = Math.min(player.maxChakra, player.chakra + 4);

  if(enemy.hp <= 0){
    enemy.alive = false;

    spawnParticles(enemy.x+enemy.w/2, enemy.y+enemy.h/2, "#eaffdd", 42, 8);

    chakraOrbs.push({
      x:enemy.x+enemy.w/2,
      y:enemy.y,
      w:16,
      h:16,
      vy:-4,
      collected:false
    });

    if(enemy.type !== "boss"){
      gainSusanooEnergy(28);
    }

    if(enemy.type === "boss"){
      victory = true;
    }
  }
}

function swordAttack(){
  if(player.attackCooldown > 0) return;

  player.attackCooldown = player.susanooActive ? 17 : 24;
  player.attacking = 11;

  const reach = player.susanooActive ? 118 : 72;
  const height = player.susanooActive ? 70 : 40;
  const damage = player.susanooActive ? 3 : 1;

  const slash = {
    x: player.x + (player.facing === 1 ? player.w : -reach),
    y: player.y + (player.susanooActive ? -4 : 14),
    w: reach,
    h: height,
    life: 10,
    facing: player.facing,
    susanoo: player.susanooActive
  };

  slashEffects.push(slash);

  for(let enemy of enemies){
    if(!enemy.alive) continue;
    if(rectsCollide(slash, enemy)){
      damageEnemy(enemy, damage, player.susanooActive ? "#b47cff" : "#d6d9ff");
    }
  }
}

function dash(){
  if(player.dashCooldown > 0 || player.dashing > 0) return;

  player.dashing = 10;
  player.dashCooldown = 46;
  player.vx = player.facing * 15;

  spawnParticles(player.x+player.w/2, player.y+player.h/2, "#9d8cff", 22, 6);
}

function castAmaterasu(){
  if(player.amaterasuCooldown > 0 || player.chakra < 18) return;

  player.chakra -= 18;
  player.amaterasuCooldown = 65;

  blackFlames.push({
    x:player.x+player.w/2+player.facing*78,
    y:player.y+34,
    w:58,
    h:66,
    vx:player.facing*6,
    life:145,
    damageTick:0,
    enemySpore:false
  });

  spawnParticles(player.x+player.facing*70, player.y+34, "#08080c", 28, 6);
  spawnParticles(player.x+player.facing*70, player.y+34, "#6411a8", 12, 4);
}

function castKirin(){
  if(player.kirinCooldown > 0 || player.chakra < 60) return;

  player.chakra -= 60;
  player.kirinCooldown = 210;
  stormFlash = 18;

  let targetX = player.x + player.facing * 270;
  targetX = Math.max(90, Math.min(world.width-90, targetX));

  lightningBolts.push({
    x:targetX,
    y:0,
    radius:165,
    life:30,
    hitDone:false
  });

  spawnParticles(targetX, 310, "#bff7ff", 60, 9);
}

function activateSusanoo(){
  if(player.susanooActive) return;
  if(player.susanooEnergy < player.susanooMax) return;

  player.susanooActive = true;
  player.susanooTimer = player.susanooDuration;
  player.susanooEnergy = 0;

  stormFlash = 10;
  spawnParticles(player.x+player.w/2, player.y+player.h/2, "#a970ff", 80, 10);
  addDamageText(player.x-20, player.y-20, "SUSANOO!", "#cfa7ff");
}

function damagePlayer(amount, knockbackDir){
  if(player.invincible > 0 || gameOver) return;

  if(player.susanooActive){
    amount = Math.max(1, amount - 1);
  }

  player.hp -= amount;
  player.invincible = player.susanooActive ? 50 : 75;
  player.vx = knockbackDir * 8;
  player.vy = -8;

  spawnParticles(player.x+player.w/2, player.y+player.h/2, "#ff4d6d", 24, 6);
  addDamageText(player.x, player.y-10, "-" + amount, "#ff4d6d");

  if(player.hp <= 0){
    player.hp = 0;
    gameOver = true;
  }
}

function maybeStartBossCutscene(){
  if(bossCutscene.done || bossCutscene.active) return;

  const boss = enemies.find(e => e.type === "boss");
  if(!boss || !boss.alive) return;

  if(player.x > 2660){
    bossCutscene.active = true;
    bossCutscene.timer = 0;
    player.vx = 0;
    player.vy = 0;
  }
}

function updateBossCutscene(){
  if(!bossCutscene.active) return;

  bossCutscene.timer++;

  const boss = enemies.find(e => e.type === "boss");

  if(boss){
    const targetCam = boss.x + boss.w/2 - WIDTH/2;
    camera.x += (Math.max(0, Math.min(world.width-WIDTH, targetCam)) - camera.x) * 0.04;

    if(bossCutscene.timer % 18 === 0){
      spawnParticles(boss.x+boss.w/2, boss.y+boss.h/2, "#dfffd0", 10, 4);
    }
  }

  if(bossCutscene.timer > 230){
    bossCutscene.active = false;
    bossCutscene.done = true;
    stormFlash = 8;
  }
}

function updatePlayer(){
  if(gameOver || victory || bossCutscene.active) return;

  player.chakra = Math.min(player.maxChakra, player.chakra + 0.035);

  if(player.susanooActive){
    player.susanooTimer--;
    if(player.susanooTimer <= 0){
      player.susanooActive = false;
      spawnParticles(player.x+player.w/2, player.y+player.h/2, "#a970ff", 30, 7);
    }
  }

  if(player.attackCooldown > 0) player.attackCooldown--;
  if(player.attacking > 0) player.attacking--;
  if(player.dashCooldown > 0) player.dashCooldown--;
  if(player.dashing > 0) player.dashing--;
  if(player.invincible > 0) player.invincible--;
  if(player.amaterasuCooldown > 0) player.amaterasuCooldown--;
  if(player.kirinCooldown > 0) player.kirinCooldown--;

  let moving = false;

  if(keys["a"]){
    player.vx -= player.speed;
    player.facing = -1;
    moving = true;
  }

  if(keys["d"]){
    player.vx += player.speed;
    player.facing = 1;
    moving = true;
  }

  if(!moving && player.dashing <= 0){
    player.vx *= friction;
  }

  if(player.vx > player.maxSpeed && player.dashing <= 0) player.vx = player.maxSpeed;
  if(player.vx < -player.maxSpeed && player.dashing <= 0) player.vx = -player.maxSpeed;

  const jumpPressed = pressed["w"] || pressed[" "];

  if(jumpPressed && player.grounded){
    player.vy = player.jumpPower;
    player.grounded = false;
    player.canDoubleJump = true;
    spawnParticles(player.x+player.w/2, player.y+player.h, "#6d6a56", 10, 4);
  } else if(jumpPressed && player.canDoubleJump && !player.grounded){
    player.vy = player.jumpPower * 0.86;
    player.canDoubleJump = false;
    spawnParticles(player.x+player.w/2, player.y+player.h/2, "#b6a6ff", 16, 5);
  }

  if(pressed["j"]) swordAttack();
  if(pressed["k"]) dash();
  if(pressed["u"]) castAmaterasu();
  if(pressed["i"]) castKirin();
  if(pressed["o"]) activateSusanoo();

  if(player.dashing <= 0){
    player.vy += gravity;
  } else {
    player.vy *= 0.25;
  }

  player.x += player.vx;
  handleHorizontalCollision();

  player.y += player.vy;
  handleVerticalCollision();

  player.x = Math.max(0, Math.min(world.width-player.w, player.x));

  if(player.y > HEIGHT+230){
    damagePlayer(2, -player.facing);
    player.x = 80;
    player.y = 250;
    player.vx = 0;
    player.vy = 0;
  }

  for(let spike of spikes){
    if(rectsCollide(player, spike)){
      damagePlayer(1, player.x < spike.x ? -1 : 1);
    }
  }

  maybeStartBossCutscene();
}

function handleHorizontalCollision(){
  for(let p of platforms){
    if(rectsCollide(player,p)){
      if(player.vx > 0) player.x = p.x - player.w;
      else if(player.vx < 0) player.x = p.x + p.w;
      player.vx = 0;
    }
  }
}

function handleVerticalCollision(){
  player.grounded = false;

  for(let p of platforms){
    if(rectsCollide(player,p)){
      if(player.vy > 0){
        player.y = p.y - player.h;
        player.vy = 0;
        player.grounded = true;
        player.canDoubleJump = true;
      } else if(player.vy < 0){
        player.y = p.y + p.h;
        player.vy = 0;
      }
    }
  }
}

function updateEnemies(){
  if(gameOver || victory || bossCutscene.active) return;

  for(let enemy of enemies){
    if(!enemy.alive) continue;

    if(enemy.hurt > 0) enemy.hurt--;
    if(enemy.susanooHitCooldown > 0) enemy.susanooHitCooldown--;

    if(enemy.burn > 0){
      enemy.burn--;
      if(enemy.burn % 24 === 0){
        damageEnemy(enemy, 1, "#111111", false);
      }
      spawnParticles(enemy.x+enemy.w/2, enemy.y+enemy.h/2, "#050509", 2, 2.5);
    }

    enemy.x += enemy.vx;

    if(enemy.x < enemy.minX || enemy.x+enemy.w > enemy.maxX){
      enemy.vx *= -1;
    }

    let dx = (player.x+player.w/2) - (enemy.x+enemy.w/2);
    let distance = Math.abs(dx);

    if((enemy.type === "fast" || enemy.type === "boss") && distance < 330){
      enemy.vx += Math.sign(dx)*0.035;
      enemy.vx = Math.max(-2.35, Math.min(2.35, enemy.vx));
    }

    if(enemy.type === "spore" || enemy.type === "boss"){
      enemy.sporeCooldown--;
      if(enemy.sporeCooldown <= 0 && distance < 380){
        enemy.sporeCooldown = enemy.type === "boss" ? 78 : 130;

        blackFlames.push({
          x:enemy.x+enemy.w/2,
          y:enemy.y+enemy.h/2,
          w:36,
          h:36,
          vx:Math.sign(dx)*3.25,
          life:80,
          damageTick:0,
          enemySpore:true
        });
      }
    }

    if(player.susanooActive && enemy.susanooHitCooldown <= 0){
      const aura = {
        x:player.x-55,
        y:player.y-80,
        w:player.w+110,
        h:player.h+110
      };

      if(rectsCollide(aura, enemy)){
        enemy.susanooHitCooldown = 45;
        damageEnemy(enemy, 2, "#b47cff", false);
      }
    }

    if(rectsCollide(player, enemy)){
      damagePlayer(enemy.type === "boss" ? 2 : 1, player.x < enemy.x ? -1 : 1);
    }
  }
}

function updateBlackFlames(){
  blackFlames = blackFlames.filter(f => f.life > 0);

  for(let f of blackFlames){
    f.x += f.vx;
    f.life--;

    const box = {
      x:f.x-f.w/2,
      y:f.y-f.h/2,
      w:f.w,
      h:f.h
    };

    if(!f.enemySpore){
      f.damageTick++;

      for(let enemy of enemies){
        if(!enemy.alive) continue;

        if(rectsCollide(box,enemy)){
          enemy.burn = Math.max(enemy.burn, 100);

          if(f.damageTick % 18 === 0){
            damageEnemy(enemy, 1, "#111111", false);
          }
        }
      }
    } else {
      if(rectsCollide(player,box)){
        damagePlayer(1, player.x < f.x ? -1 : 1);
        f.life = 0;
      }
    }
  }
}

function updateLightning(){
  lightningBolts = lightningBolts.filter(l => l.life > 0);

  for(let l of lightningBolts){
    l.life--;

    if(!l.hitDone && l.life < 21){
      l.hitDone = true;

      for(let enemy of enemies){
        if(!enemy.alive) continue;

        let cx = enemy.x+enemy.w/2;
        let cy = enemy.y+enemy.h/2;
        let dx = cx-l.x;
        let dy = cy-330;
        let dist = Math.sqrt(dx*dx + dy*dy);

        if(dist < l.radius){
          damageEnemy(enemy, enemy.type === "boss" ? 7 : 4, "#bff7ff", false);
          enemy.burn = Math.max(enemy.burn, 36);
        }
      }
    }
  }

  if(stormFlash > 0) stormFlash--;
}

function updateChakraOrbs(){
  for(let orb of chakraOrbs){
    if(orb.collected) continue;

    orb.vy += 0.2;
    orb.y += orb.vy;

    if(orb.y > 452){
      orb.y = 452;
      orb.vy *= -0.35;
    }

    let pickupBox = {
      x:orb.x-14,
      y:orb.y-14,
      w:44,
      h:44
    };

    if(rectsCollide(player,pickupBox)){
      orb.collected = true;
      player.chakra = Math.min(player.maxChakra, player.chakra+24);
      spawnParticles(orb.x, orb.y, "#b6a6ff", 20, 5);
    }
  }
}

function updateParticles(){
  particles = particles.filter(p => p.life > 0);

  for(let p of particles){
    p.x += p.vx;
    p.y += p.vy;
    p.vy += 0.13;
    p.life--;
  }

  slashEffects = slashEffects.filter(s => s.life > 0);
  for(let s of slashEffects) s.life--;

  damageTexts = damageTexts.filter(t => t.life > 0);
  for(let t of damageTexts){
    t.y += t.vy;
    t.life--;
  }

  for(let leaf of leaves){
    leaf.x += leaf.vx;
    leaf.y += leaf.vy;

    if(leaf.x < camera.x-120){
      leaf.x = camera.x + WIDTH + Math.random()*220;
      leaf.y = Math.random()*HEIGHT;
    }

    if(leaf.y > HEIGHT){
      leaf.y = -20;
    }
  }
}

function updateCamera(){
  if(bossCutscene.active) return;

  camera.x = player.x + player.w/2 - WIDTH/2;
  camera.x = Math.max(0, Math.min(world.width-WIDTH, camera.x));
}

function drawBackground(){
  let gradient = ctx.createLinearGradient(0,0,0,HEIGHT);
  gradient.addColorStop(0,"#060716");
  gradient.addColorStop(0.55,"#111026");
  gradient.addColorStop(1,"#17161d");

  ctx.fillStyle = gradient;
  ctx.fillRect(0,0,WIDTH,HEIGHT);

  // lua
  ctx.fillStyle = "rgba(230,220,255,0.82)";
  ctx.beginPath();
  ctx.arc(820,70,36,0,Math.PI*2);
  ctx.fill();

  ctx.fillStyle = "rgba(80,65,125,0.35)";
  ctx.beginPath();
  ctx.arc(838,60,36,0,Math.PI*2);
  ctx.fill();

  // montanhas
  ctx.save();
  ctx.translate(-camera.x*0.15,0);

  for(let i=0;i<10;i++){
    let x = i*420;
    ctx.fillStyle = i%2===0 ? "#15172a" : "#101224";
    ctx.beginPath();
    ctx.moveTo(x-120,470);
    ctx.lineTo(x+135,165);
    ctx.lineTo(x+390,470);
    ctx.closePath();
    ctx.fill();
  }

  ctx.restore();

  // vila ao fundo
  ctx.save();
  ctx.translate(-camera.x*0.32,0);

  for(let i=0;i<24;i++){
    let x = i*170;
    let y = 240 + (i%3)*16;

    ctx.fillStyle = i%2===0 ? "#18182b" : "#141426";
    ctx.fillRect(x,y,86,230);

    ctx.fillStyle = "#2a2744";
    ctx.beginPath();
    ctx.moveTo(x-8,y);
    ctx.lineTo(x+43,y-38);
    ctx.lineTo(x+94,y);
    ctx.closePath();
    ctx.fill();

    ctx.fillStyle = "rgba(230,180,70,0.17)";
    ctx.fillRect(x+20,y+55,12,25);
    ctx.fillRect(x+54,y+112,12,25);
  }

  ctx.restore();

  // floresta
  ctx.save();
  ctx.translate(-camera.x*0.48,0);

  for(let i=0;i<42;i++){
    let x = i*95;

    ctx.fillStyle = "#101610";
    ctx.fillRect(x+20,280,16,195);

    ctx.fillStyle = "#172514";
    ctx.beginPath();
    ctx.arc(x+26,260,44,0,Math.PI*2);
    ctx.fill();

    ctx.fillStyle = "#1f331b";
    ctx.beginPath();
    ctx.arc(x+6,292,35,0,Math.PI*2);
    ctx.arc(x+48,292,36,0,Math.PI*2);
    ctx.fill();
  }

  ctx.restore();

  // folhas
  ctx.save();
  ctx.translate(-camera.x*0.15,0);

  for(let leaf of leaves){
    ctx.fillStyle = leaf.color;
    ctx.fillRect(leaf.x, leaf.y, leaf.size, leaf.size+2);
  }

  ctx.restore();

  if(stormFlash > 0){
    ctx.fillStyle = "rgba(190,245,255," + (stormFlash/34) + ")";
    ctx.fillRect(0,0,WIDTH,HEIGHT);
  }
}

function drawPlatforms(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let p of platforms){
    ctx.fillStyle = "#303035";
    ctx.fillRect(p.x,p.y,p.w,p.h);

    ctx.fillStyle = "#656145";
    ctx.fillRect(p.x,p.y,p.w,8);

    ctx.fillStyle = "#252421";
    ctx.fillRect(p.x,p.y+p.h-13,p.w,13);

    ctx.fillStyle = "#405c2d";
    for(let i=0;i<p.w;i+=16){
      ctx.fillRect(p.x+i,p.y-5,10,7);
    }
  }

  // portões Torii
  for(let gateX of [990,1900,2800]){
    ctx.fillStyle = "#8c2525";
    ctx.fillRect(gateX,270,20,190);
    ctx.fillRect(gateX+104,270,20,190);
    ctx.fillRect(gateX-24,258,170,19);
    ctx.fillRect(gateX-45,232,215,22);
    ctx.fillStyle = "#2c1111";
    ctx.fillRect(gateX-55,224,235,9);
  }

  // espinhos
  for(let spike of spikes){
    ctx.fillStyle = "#5a332a";
    let count = Math.floor(spike.w/18);
    for(let i=0;i<count;i++){
      ctx.beginPath();
      ctx.moveTo(spike.x+i*18, spike.y+spike.h);
      ctx.lineTo(spike.x+i*18+9, spike.y);
      ctx.lineTo(spike.x+i*18+18, spike.y+spike.h);
      ctx.closePath();
      ctx.fill();
    }
  }

  ctx.restore();
}

function drawSusanoo(){
  if(!player.susanooActive) return;

  ctx.save();
  ctx.translate(-camera.x,0);

  let x = player.x + player.w/2;
  let y = player.y + player.h/2;

  ctx.globalAlpha = 0.38;

  // aura gigante
  ctx.fillStyle = "#7d3cff";
  ctx.beginPath();
  ctx.ellipse(x, y+5, 72, 105, 0, 0, Math.PI*2);
  ctx.fill();

  ctx.globalAlpha = 0.55;

  // esqueleto/cabeça do Susanoo
  ctx.strokeStyle = "#d3a6ff";
  ctx.lineWidth = 5;

  ctx.beginPath();
  ctx.arc(x, y-58, 37, 0, Math.PI*2);
  ctx.stroke();

  // olhos do avatar
  ctx.fillStyle = "#f1dfff";
  ctx.fillRect(x-18, y-65, 11, 7);
  ctx.fillRect(x+7, y-65, 11, 7);

  // costelas
  ctx.strokeStyle = "#c084ff";
  ctx.lineWidth = 4;

  for(let i=0;i<4;i++){
    ctx.beginPath();
    ctx.arc(x, y-15+i*22, 55-i*5, Math.PI*1.08, Math.PI*1.92);
    ctx.stroke();
  }

  // braços
  ctx.beginPath();
  ctx.moveTo(x-45,y-20);
  ctx.lineTo(x-95,y+25);
  ctx.moveTo(x+45,y-20);
  ctx.lineTo(x+95,y+25);
  ctx.stroke();

  // espada espiritual
  ctx.strokeStyle = "#e9d5ff";
  ctx.lineWidth = 6;
  ctx.beginPath();
  ctx.moveTo(x + player.facing*35, y-40);
  ctx.lineTo(x + player.facing*120, y-95);
  ctx.stroke();

  ctx.globalAlpha = 1;
  ctx.restore();
}

function drawSasuke(){
  ctx.save();
  ctx.translate(-camera.x,0);

  if(player.invincible > 0 && Math.floor(player.invincible/5)%2===0){
    ctx.globalAlpha = 0.45;
  }

  let x = player.x;
  let y = player.y;

  // sombra
  ctx.fillStyle = "rgba(0,0,0,0.55)";
  ctx.beginPath();
  ctx.ellipse(x+player.w/2, y+player.h+5, 29, 8, 0,0,Math.PI*2);
  ctx.fill();

  // aura chakra
  if(player.chakra > 30 || player.susanooActive){
    ctx.strokeStyle = player.susanooActive ? "rgba(180,110,255,0.75)" : "rgba(140,90,255,0.35)";
    ctx.lineWidth = player.susanooActive ? 3 : 2;
    ctx.beginPath();
    ctx.arc(x+player.w/2, y+player.h/2, 38+Math.sin(Date.now()/120)*4, 0, Math.PI*2);
    ctx.stroke();
  }

  // capa/parte escura
  ctx.fillStyle = "#11111d";
  ctx.beginPath();
  ctx.moveTo(x+5,y+22);
  ctx.lineTo(x-10,y+68);
  ctx.lineTo(x+29,y+66);
  ctx.lineTo(x+34,y+22);
  ctx.closePath();
  ctx.fill();

  // roupa clara
  rr(x+8,y+23,22,34,4,"#d7d4e6");

  // parte roxa
  rr(x+5,y+36,28,22,4,"#35304f");

  // corda na cintura
  ctx.strokeStyle = "#b9adc8";
  ctx.lineWidth = 4;
  ctx.beginPath();
  ctx.moveTo(x+4,y+45);
  ctx.bezierCurveTo(x+10,y+52,x+25,y+52,x+34,y+45);
  ctx.stroke();

  ctx.fillStyle = "#b9adc8";
  ctx.beginPath();
  ctx.arc(x+19,y+49,5,0,Math.PI*2);
  ctx.fill();

  // pernas
  rr(x+8,y+56,8,14,2,"#171522");
  rr(x+23,y+56,8,14,2,"#171522");

  // cabeça/pele estilizada sem detalhe realista
  rr(x+4,y+4,30,26,7,"#ddddea");

  // cabelo preto espetado mais marcante
  ctx.fillStyle = "#05050a";

  ctx.beginPath();
  ctx.moveTo(x+3,y+10);
  ctx.lineTo(x-7,y-8);
  ctx.lineTo(x+6,y+0);
  ctx.lineTo(x+10,y-16);
  ctx.lineTo(x+17,y+0);
  ctx.lineTo(x+25,y-15);
  ctx.lineTo(x+29,y+2);
  ctx.lineTo(x+40,y-6);
  ctx.lineTo(x+35,y+15);
  ctx.lineTo(x+23,y+8);
  ctx.lineTo(x+11,y+12);
  ctx.closePath();
  ctx.fill();

  // olhos: sharingan e rinnegan estilizados
  ctx.fillStyle = "#d71732";
  ctx.fillRect(x+10,y+15,6,5);

  ctx.fillStyle = "#b6a6ff";
  ctx.beginPath();
  ctx.arc(x+25,y+17,4,0,Math.PI*2);
  ctx.fill();

  ctx.strokeStyle = "#5e45b8";
  ctx.lineWidth = 1;
  ctx.beginPath();
  ctx.arc(x+25,y+17,7,0,Math.PI*2);
  ctx.stroke();

  ctx.beginPath();
  ctx.arc(x+25,y+17,4,0,Math.PI*2);
  ctx.stroke();

  // espada nas costas/mão
  ctx.strokeStyle = "#cfd4ff";
  ctx.lineWidth = 3;
  ctx.beginPath();
  ctx.moveTo(x+player.w/2, y+34);
  ctx.lineTo(x+player.w/2 + player.facing*32, y+18);
  ctx.stroke();

  ctx.fillStyle = "#534d73";
  ctx.fillRect(x+player.w/2-2, y+32, 4, 10);

  // dash trail
  if(player.dashing > 0){
    ctx.strokeStyle = "rgba(160,145,255,0.7)";
    ctx.lineWidth = 4;
    ctx.beginPath();
    ctx.moveTo(x - player.facing*25, y+28);
    ctx.lineTo(x - player.facing*88, y+40);
    ctx.stroke();
  }

  ctx.globalAlpha = 1;
  ctx.restore();
}

function drawEnemies(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let enemy of enemies){
    if(!enemy.alive) continue;

    let x = enemy.x;
    let y = enemy.y;

    ctx.globalAlpha = enemy.hurt > 0 ? 0.55 : 1;

    // sombra
    ctx.fillStyle = "rgba(0,0,0,0.5)";
    ctx.beginPath();
    ctx.ellipse(x+enemy.w/2, y+enemy.h+6, enemy.w/1.15, 8, 0,0,Math.PI*2);
    ctx.fill();

    // corpo branco
    rr(x,y,enemy.w,enemy.h,16, enemy.burn > 0 ? "#c9c3d9" : "#f0f3e5");

    // folhas laterais estilo Zetsu
    ctx.fillStyle = "#567f3c";

    ctx.beginPath();
    ctx.moveTo(x+enemy.w/2,y+6);
    ctx.lineTo(x-18,y+enemy.h*0.26);
    ctx.lineTo(x+6,y+enemy.h*0.58);
    ctx.lineTo(x+enemy.w/2,y+enemy.h*0.45);
    ctx.closePath();
    ctx.fill();

    ctx.beginPath();
    ctx.moveTo(x+enemy.w/2,y+6);
    ctx.lineTo(x+enemy.w+18,y+enemy.h*0.26);
    ctx.lineTo(x+enemy.w-6,y+enemy.h*0.58);
    ctx.lineTo(x+enemy.w/2,y+enemy.h*0.45);
    ctx.closePath();
    ctx.fill();

    // veias verdes
    ctx.strokeStyle = "#365d2a";
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(x+enemy.w/2,y+8);
    ctx.lineTo(x+enemy.w/2,y+enemy.h-8);
    ctx.moveTo(x+enemy.w/2,y+24);
    ctx.lineTo(x+8,y+enemy.h*0.55);
    ctx.moveTo(x+enemy.w/2,y+24);
    ctx.lineTo(x+enemy.w-8,y+enemy.h*0.55);
    ctx.stroke();

    // rosto
    ctx.fillStyle = "#101010";
    ctx.fillRect(x+enemy.w*0.28, y+enemy.h*0.33, 5, 7);
    ctx.fillRect(x+enemy.w*0.62, y+enemy.h*0.33, 5, 7);

    ctx.strokeStyle = "#101010";
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.arc(x+enemy.w/2, y+enemy.h*0.52, enemy.w*0.16, 0, Math.PI);
    ctx.stroke();

    // boss aura
    if(enemy.type === "boss"){
      ctx.strokeStyle = "#dfffd0";
      ctx.lineWidth = 4;
      ctx.beginPath();
      ctx.arc(x+enemy.w/2, y+enemy.h/2, 62+Math.sin(Date.now()/150)*5, 0, Math.PI*2);
      ctx.stroke();

      ctx.fillStyle = "rgba(220,255,180,0.18)";
      ctx.beginPath();
      ctx.arc(x+enemy.w/2, y+enemy.h/2, 72, 0, Math.PI*2);
      ctx.fill();
    }

    // fogo negro no corpo
    if(enemy.burn > 0){
      ctx.fillStyle = "#050509";
      for(let i=0;i<6;i++){
        let fx = x+6+i*(enemy.w/6);
        let fy = y+enemy.h-8-Math.random()*30;

        ctx.beginPath();
        ctx.moveTo(fx,fy+18);
        ctx.lineTo(fx+6,fy-12);
        ctx.lineTo(fx+12,fy+18);
        ctx.closePath();
        ctx.fill();
      }
    }

    // vida
    ctx.globalAlpha = 1;
    ctx.fillStyle = "#111";
    ctx.fillRect(x,y-14,enemy.w,6);

    ctx.fillStyle = enemy.type === "boss" ? "#c7ff9e" : "#e9ffd4";
    ctx.fillRect(x,y-14,enemy.w*(enemy.hp/enemy.maxHp),6);
  }

  ctx.restore();
}

function drawSlashEffects(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let s of slashEffects){
    ctx.globalAlpha = s.life/10;
    ctx.strokeStyle = s.susanoo ? "#cfa7ff" : "#d6d9ff";
    ctx.lineWidth = s.susanoo ? 8 : 5;
    ctx.beginPath();

    if(s.facing === 1){
      ctx.arc(s.x+18, s.y+20, s.susanoo ? 64 : 40, -0.9, 0.85);
    } else {
      ctx.arc(s.x+s.w-18, s.y+20, s.susanoo ? 64 : 40, Math.PI-0.85, Math.PI+0.9);
    }

    ctx.stroke();
  }

  ctx.globalAlpha = 1;
  ctx.restore();
}

function drawBlackFlames(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let f of blackFlames){
    ctx.globalAlpha = Math.max(0.25, f.life/145);

    for(let i=0;i<9;i++){
      let h = (f.enemySpore ? 24 : 44) + Math.random()*18;
      let w = 10 + Math.random()*7;
      let px = f.x - f.w/2 + i*(f.w/8);
      let baseY = f.y + f.h/2;

      ctx.fillStyle = f.enemySpore ? "#dfffd0" : "#050509";
      ctx.beginPath();
      ctx.moveTo(px,baseY);
      ctx.lineTo(px+w/2,baseY-h);
      ctx.lineTo(px+w,baseY);
      ctx.closePath();
      ctx.fill();

      if(!f.enemySpore){
        ctx.fillStyle = "rgba(93,17,168,0.45)";
        ctx.beginPath();
        ctx.moveTo(px+2,baseY);
        ctx.lineTo(px+w/2,baseY-h*0.65);
        ctx.lineTo(px+w-2,baseY);
        ctx.closePath();
        ctx.fill();
      }
    }

    ctx.globalAlpha = 1;
  }

  ctx.restore();
}

function drawLightning(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let l of lightningBolts){
    let alpha = l.life/30;
    ctx.globalAlpha = alpha;

    ctx.strokeStyle = "#e6ffff";
    ctx.lineWidth = 7;
    ctx.beginPath();

    let x = l.x;
    let y = 0;

    ctx.moveTo(x,y);

    for(let i=0;i<11;i++){
      x += (Math.random()-0.5)*80;
      y += 44;
      ctx.lineTo(x,y);
    }

    ctx.stroke();

    ctx.strokeStyle = "rgba(120,210,255,0.8)";
    ctx.lineWidth = 2;
    ctx.beginPath();

    x = l.x;
    y = 0;
    ctx.moveTo(x,y);

    for(let i=0;i<13;i++){
      x += (Math.random()-0.5)*100;
      y += 38;
      ctx.lineTo(x,y);
    }

    ctx.stroke();

    ctx.fillStyle = "rgba(190,247,255,0.18)";
    ctx.beginPath();
    ctx.arc(l.x,330,l.radius,0,Math.PI*2);
    ctx.fill();

    ctx.globalAlpha = 1;
  }

  ctx.restore();
}

function drawChakraOrbs(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let orb of chakraOrbs){
    if(orb.collected) continue;

    ctx.fillStyle = "#b6a6ff";
    ctx.beginPath();
    ctx.arc(orb.x,orb.y,8,0,Math.PI*2);
    ctx.fill();

    ctx.strokeStyle = "rgba(185,165,255,0.55)";
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.arc(orb.x,orb.y,15,0,Math.PI*2);
    ctx.stroke();
  }

  ctx.restore();
}

function drawParticles(){
  ctx.save();
  ctx.translate(-camera.x,0);

  for(let p of particles){
    ctx.globalAlpha = Math.max(0,p.life/56);
    ctx.fillStyle = p.color;
    ctx.fillRect(p.x,p.y,p.size,p.size);
  }

  ctx.globalAlpha = 1;
  ctx.restore();
}

function drawDamageTexts(){
  ctx.save();
  ctx.translate(-camera.x,0);

  ctx.font = "bold 16px Arial";

  for(let t of damageTexts){
    ctx.globalAlpha = t.life/45;
    ctx.fillStyle = t.color;
    ctx.fillText(t.text,t.x,t.y);
  }

  ctx.globalAlpha = 1;
  ctx.restore();
}

function drawHUD(){
  ctx.fillStyle = "rgba(0,0,0,0.52)";
  ctx.fillRect(15,15,365,132);

  // vida
  for(let i=0;i<player.maxHp;i++){
    ctx.fillStyle = i < player.hp ? "#e8e6ff" : "#33313f";
    ctx.beginPath();
    ctx.arc(36+i*24,40,9,0,Math.PI*2);
    ctx.fill();

    ctx.strokeStyle = "#111";
    ctx.stroke();
  }

  // chakra
  ctx.fillStyle = "#111";
  ctx.fillRect(30,68,230,13);

  ctx.fillStyle = "#8f7dff";
  ctx.fillRect(30,68,230*(player.chakra/player.maxChakra),13);

  ctx.strokeStyle = "#5b50a8";
  ctx.strokeRect(30,68,230,13);

  ctx.fillStyle = "#ddd";
  ctx.font = "12px Arial";
  ctx.fillText("CHAKRA",270,79);

  // Susanoo energy
  ctx.fillStyle = "#111";
  ctx.fillRect(30,95,230,14);

  let sRatio = player.susanooActive
    ? player.susanooTimer/player.susanooDuration
    : player.susanooEnergy/player.susanooMax;

  ctx.fillStyle = player.susanooActive ? "#cfa7ff" : "#7d3cff";
  ctx.fillRect(30,95,230*sRatio,14);

  ctx.strokeStyle = "#9f70ff";
  ctx.strokeRect(30,95,230,14);

  ctx.fillStyle = "#ddd";
  ctx.fillText(player.susanooActive ? "SUSANOO ATIVO" : "ENERGIA SUSANOO",270,106);

  ctx.font = "12px Arial";

  ctx.fillStyle = player.amaterasuCooldown <= 0 && player.chakra >= 18 ? "#d7c6ff" : "#777";
  ctx.fillText("U Amaterasu",30,130);

  ctx.fillStyle = player.kirinCooldown <= 0 && player.chakra >= 60 ? "#bff7ff" : "#777";
  ctx.fillText("I Kirin",132,130);

  ctx.fillStyle = player.susanooEnergy >= player.susanooMax && !player.susanooActive ? "#cfa7ff" : "#777";
  ctx.fillText("O Susanoo",205,130);

  ctx.fillStyle = "#aaa";
  ctx.font = "13px Arial";
  ctx.fillText("Derrote Zetsus para carregar o Susanoo", 660, 28);
}

function drawMiniMap(){
  ctx.fillStyle = "rgba(0,0,0,0.38)";
  ctx.fillRect(642,492,285,22);

  ctx.strokeStyle = "#555";
  ctx.strokeRect(642,492,285,22);

  let px = 642 + (player.x/world.width)*285;

  ctx.fillStyle = "#b6a6ff";
  ctx.fillRect(px,495,5,16);

  for(let enemy of enemies){
    if(!enemy.alive) continue;

    let ex = 642 + (enemy.x/world.width)*285;
    ctx.fillStyle = enemy.type === "boss" ? "#c7ff9e" : "#f4f5e8";
    ctx.fillRect(ex,500,4,8);
  }
}

function drawCutscene(){
  if(!bossCutscene.active) return;

  ctx.fillStyle = "rgba(0,0,0,0.42)";
  ctx.fillRect(0,0,WIDTH,HEIGHT);

  ctx.fillStyle = "rgba(0,0,0,0.82)";
  ctx.fillRect(0,HEIGHT-135,WIDTH,135);

  ctx.strokeStyle = "#c7ff9e";
  ctx.lineWidth = 2;
  ctx.strokeRect(24,HEIGHT-120,WIDTH-48,98);

  ctx.fillStyle = "#e9ffd4";
  ctx.font = "bold 22px Arial";

  let text = "";

  if(bossCutscene.timer < 75){
    text = "Algo está emergindo da floresta...";
  } else if(bossCutscene.timer < 155){
    text = "Zetsu Maior: Você não passará daqui.";
  } else {
    text = "Sasuke: Então eu vou remover você do caminho.";
  }

  ctx.fillText(text, 52, HEIGHT-76);

  ctx.fillStyle = "#aaa";
  ctx.font = "14px Arial";
  ctx.fillText("Cutscene do chefe", 52, HEIGHT-46);
}

function drawMessages(){
  if(gameOver){
    ctx.fillStyle = "rgba(0,0,0,0.74)";
    ctx.fillRect(0,0,WIDTH,HEIGHT);

    ctx.fillStyle = "#d7c6ff";
    ctx.font = "bold 46px Arial";
    ctx.textAlign = "center";
    ctx.fillText("VOCÊ FOI DERROTADO", WIDTH/2, HEIGHT/2-25);

    ctx.fillStyle = "#ddd";
    ctx.font = "22px Arial";
    ctx.fillText("Pressione R para reiniciar", WIDTH/2, HEIGHT/2+25);
    ctx.textAlign = "left";
  }

  if(victory){
    ctx.fillStyle = "rgba(0,0,0,0.70)";
    ctx.fillRect(0,0,WIDTH,HEIGHT);

    ctx.fillStyle = "#bff7ff";
    ctx.font = "bold 46px Arial";
    ctx.textAlign = "center";
    ctx.fillText("ZETSU MAIOR DERROTADO!", WIDTH/2, HEIGHT/2-35);

    ctx.fillStyle = "#ddd";
    ctx.font = "22px Arial";
    ctx.fillText("Sasuke dominou o campo de batalha", WIDTH/2, HEIGHT/2+10);
    ctx.fillText("Pressione R para jogar novamente", WIDTH/2, HEIGHT/2+45);
    ctx.textAlign = "left";
  }
}

function update(){
  updateBossCutscene();

  updatePlayer();
  updateEnemies();
  updateBlackFlames();
  updateLightning();
  updateChakraOrbs();
  updateParticles();
  updateCamera();

  pressed = {};
}

function draw(){
  ctx.clearRect(0,0,WIDTH,HEIGHT);

  drawBackground();
  drawPlatforms();
  drawChakraOrbs();
  drawBlackFlames();
  drawLightning();
  drawSlashEffects();

  // Susanoo atrás do personagem, mas na frente do cenário
  drawSusanoo();

  drawEnemies();
  drawSasuke();
  drawParticles();
  drawDamageTexts();
  drawHUD();
  drawMiniMap();
  drawCutscene();
  drawMessages();
}

function loop(){
  update();
  draw();
  requestAnimationFrame(loop);
}

resetGame();
loop();
</script>
""")